In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# Project Scope

This agent turns messy project briefs into clear execution plans.

# Problem Statement

Teams often receive messy project briefs, meeting notes, or product requests that are hard to turn into action. This agent helps by converting unstructured input into a clear execution plan with a summary, tasks, risks, assumptions, and open questions.

# Success Criteria

The agent should:
- summarize the brief clearly
- identify the main goal
- list assumptions and missing information
- break the work into actionable tasks
- highlight risks
- produce a clean markdown plan

# Example Input

Launch a new customer onboarding flow for the website. Need it faster, less confusing, and ready by next month. Include sign-up, email verification, and basic analytics.

# Example Output Format

## Summary
## Goal
## Assumptions
## Open Questions
## Task Breakdown
## Risks
## Suggested Next Step

# Architecture

This project will use:
- one main orchestrator agent
- one analysis step to extract key facts
- one planning step to generate tasks and risks
- one review step to check quality
- one small MCP-style tool layer for sample briefs and templates

In [1]:
import re
from textwrap import shorten

def generate_execution_plan(brief: str) -> str:
    """
    Turn a messy project brief into a clean markdown execution plan.
    This is a simple baseline version before we add AI agents.
    """
    text = brief.strip()
    lower = text.lower()

    # 1) Summary: keep it short and readable
    first_sentence = re.split(r'[.!?]', text)[0].strip()
    summary = shorten(first_sentence, width=140, placeholder="...")

    # 2) Goal: guess the main intent from keywords
    if "onboarding" in lower:
        goal = "Improve the customer onboarding experience."
    elif "dashboard" in lower:
        goal = "Build or improve a dashboard."
    elif "report" in lower or "summary" in lower:
        goal = "Turn raw information into a useful report."
    else:
        goal = "Convert the request into a clear, actionable project plan."

    # 3) Assumptions: start with safe, generic assumptions
    assumptions = [
        "The request is for a real project, not just an experiment.",
        "The team wants a practical plan that can be shared with others.",
    ]

    # 4) Open questions: highlight missing info
    open_questions = [
        "What is the exact deadline?",
        "Who is the target user or customer?",
        "What tools, data, or systems are already available?",
    ]

    # 5) Task breakdown: a simple reusable structure
    tasks = [
        "Clarify the full scope and success criteria.",
        "Break the work into small deliverable steps.",
        "Implement the first version.",
        "Review the output and refine based on feedback.",
    ]

    # 6) Risks: mention common project risks
    risks = [
        "Requirements may be incomplete or unclear.",
        "The timeline may be too short for the requested scope.",
        "Important details may be missing from the brief.",
    ]

    # 7) Final markdown output
    plan = f"""# Execution Plan

## Summary
{summary}

## Goal
{goal}

## Assumptions
- {assumptions[0]}
- {assumptions[1]}

## Open Questions
- {open_questions[0]}
- {open_questions[1]}
- {open_questions[2]}

## Task Breakdown
1. {tasks[0]}
2. {tasks[1]}
3. {tasks[2]}
4. {tasks[3]}

## Risks
- {risks[0]}
- {risks[1]}
- {risks[2]}

## Suggested Next Step
Review the open questions with the user before starting implementation.
"""
    return plan


sample_brief = """
Launch a new customer onboarding flow for the website. Need it faster, less confusing, and ready by next month. Include sign-up, email verification, and basic analytics.
"""

print(generate_execution_plan(sample_brief))

# Execution Plan

## Summary
Launch a new customer onboarding flow for the website

## Goal
Improve the customer onboarding experience.

## Assumptions
- The request is for a real project, not just an experiment.
- The team wants a practical plan that can be shared with others.

## Open Questions
- What is the exact deadline?
- Who is the target user or customer?
- What tools, data, or systems are already available?

## Task Breakdown
1. Clarify the full scope and success criteria.
2. Break the work into small deliverable steps.
3. Implement the first version.
4. Review the output and refine based on feedback.

## Risks
- Requirements may be incomplete or unclear.
- The timeline may be too short for the requested scope.
- Important details may be missing from the brief.

## Suggested Next Step
Review the open questions with the user before starting implementation.



In [2]:
def review_execution_plan(plan: str) -> list[str]:
    required_sections = [
        "## Summary",
        "## Goal",
        "## Assumptions",
        "## Open Questions",
        "## Task Breakdown",
        "## Risks",
        "## Suggested Next Step",
    ]
    
    issues = []
    for section in required_sections:
        if section not in plan:
            issues.append(f"Missing section: {section}")
    
    if len(plan.strip()) < 200:
        issues.append("Plan looks too short to be useful.")
    
    return issues


test_briefs = [
    """
    Launch a new customer onboarding flow for the website. Need it faster, less confusing, and ready by next month. Include sign-up, email verification, and basic analytics.
    """,
    """
    We need a better internal report for sales leads. Team wants something simple, reliable, and easy to share in weekly meetings.
    """
]

for i, brief in enumerate(test_briefs, start=1):
    print("=" * 80)
    print(f"BRIEF {i}")
    print("-" * 80)
    plan = generate_execution_plan(brief)
    print(plan)
    
    issues = review_execution_plan(plan)
    print("\nREVIEW:")
    if issues:
        for issue in issues:
            print(f"- {issue}")
    else:
        print("- No obvious structural issues found.")

BRIEF 1
--------------------------------------------------------------------------------
# Execution Plan

## Summary
Launch a new customer onboarding flow for the website

## Goal
Improve the customer onboarding experience.

## Assumptions
- The request is for a real project, not just an experiment.
- The team wants a practical plan that can be shared with others.

## Open Questions
- What is the exact deadline?
- Who is the target user or customer?
- What tools, data, or systems are already available?

## Task Breakdown
1. Clarify the full scope and success criteria.
2. Break the work into small deliverable steps.
3. Implement the first version.
4. Review the output and refine based on feedback.

## Risks
- Requirements may be incomplete or unclear.
- The timeline may be too short for the requested scope.
- Important details may be missing from the brief.

## Suggested Next Step
Review the open questions with the user before starting implementation.


REVIEW:
- No obvious structural 

In [3]:
import re
from textwrap import shorten

def extract_facts(brief: str) -> dict:
    text = brief.strip()
    lower = text.lower()

    facts = {
        "raw_brief": text,
        "summary": shorten(re.split(r"[.!?]", text)[0].strip(), width=140, placeholder="..."),
        "goal_hint": None,
        "keywords": [],
        "possible_deadline": None,
        "possible_audience": None,
        "deliverables": [],
    }

    # keyword detection
    keyword_map = {
        "onboarding": "customer onboarding",
        "dashboard": "dashboard",
        "report": "report",
        "sales": "sales workflow",
        "email": "email workflow",
        "analytics": "analytics",
        "signup": "sign-up flow",
        "sign-up": "sign-up flow",
    }

    for key, value in keyword_map.items():
        if key in lower and value not in facts["keywords"]:
            facts["keywords"].append(value)

    # very simple goal hints
    if "onboarding" in lower:
        facts["goal_hint"] = "Improve the customer onboarding experience."
    elif "dashboard" in lower:
        facts["goal_hint"] = "Build or improve a dashboard."
    elif "report" in lower:
        facts["goal_hint"] = "Create a useful report or summary."
    else:
        facts["goal_hint"] = "Turn the request into a clear, actionable plan."

    # crude deadline detection
    deadline_patterns = [
        r"\bby next month\b",
        r"\bby tomorrow\b",
        r"\bby next week\b",
        r"\bby friday\b",
        r"\bby monday\b",
        r"\bby \d{1,2}/\d{1,2}\b",
        r"\bby \w+\b",
    ]
    for pattern in deadline_patterns:
        match = re.search(pattern, lower)
        if match:
            facts["possible_deadline"] = match.group(0)
            break

    # crude audience detection
    if "customer" in lower:
        facts["possible_audience"] = "customers"
    elif "internal" in lower or "team" in lower:
        facts["possible_audience"] = "internal team"
    elif "sales" in lower:
        facts["possible_audience"] = "sales team"

    # deliverables guess
    if "analytics" in lower:
        facts["deliverables"].append("basic analytics")
    if "email verification" in lower:
        facts["deliverables"].append("email verification")
    if "signup" in lower or "sign-up" in lower:
        facts["deliverables"].append("sign-up flow")

    return facts


def plan_from_facts(facts: dict) -> str:
    goal = facts["goal_hint"]
    summary = facts["summary"]

    assumptions = [
        "The request is for a real project, not just an experiment.",
        "The team wants a practical plan that can be shared with others.",
    ]

    open_questions = [
        "What is the exact deadline?",
        "Who is the target user or customer?",
        "What tools, data, or systems are already available?",
    ]

    tasks = [
        "Clarify the full scope and success criteria.",
        "Break the work into small deliverable steps.",
        "Implement the first version.",
        "Review the output and refine based on feedback.",
    ]

    risks = [
        "Requirements may be incomplete or unclear.",
        "The timeline may be too short for the requested scope.",
        "Important details may be missing from the brief.",
    ]

    plan = f"""# Execution Plan

## Summary
{summary}

## Goal
{goal}

## Key Facts
- Keywords detected: {", ".join(facts["keywords"]) if facts["keywords"] else "None"}
- Possible deadline: {facts["possible_deadline"] or "Not stated"}
- Possible audience: {facts["possible_audience"] or "Not stated"}
- Deliverables: {", ".join(facts["deliverables"]) if facts["deliverables"] else "Not stated"}

## Assumptions
- {assumptions[0]}
- {assumptions[1]}

## Open Questions
- {open_questions[0]}
- {open_questions[1]}
- {open_questions[2]}

## Task Breakdown
1. {tasks[0]}
2. {tasks[1]}
3. {tasks[2]}
4. {tasks[3]}

## Risks
- {risks[0]}
- {risks[1]}
- {risks[2]}

## Suggested Next Step
Review the open questions with the user before starting implementation.
"""
    return plan


def review_plan(plan: str) -> list[str]:
    required_sections = [
        "## Summary",
        "## Goal",
        "## Key Facts",
        "## Assumptions",
        "## Open Questions",
        "## Task Breakdown",
        "## Risks",
        "## Suggested Next Step",
    ]

    issues = []
    for section in required_sections:
        if section not in plan:
            issues.append(f"Missing section: {section}")

    if len(plan.strip()) < 250:
        issues.append("Plan looks too short to be useful.")

    return issues


def brief_to_plan_agent(brief: str) -> str:
    facts = extract_facts(brief)
    plan = plan_from_facts(facts)
    issues = review_plan(plan)

    if issues:
        plan += "\n## Review Notes\n"
        for issue in issues:
            plan += f"- {issue}\n"
    else:
        plan += "\n## Review Notes\n- No obvious structural issues found.\n"

    return plan


print(brief_to_plan_agent(sample_brief))

# Execution Plan

## Summary
Launch a new customer onboarding flow for the website

## Goal
Improve the customer onboarding experience.

## Key Facts
- Keywords detected: customer onboarding, email workflow, analytics, sign-up flow
- Possible deadline: by next month
- Possible audience: customers
- Deliverables: basic analytics, email verification, sign-up flow

## Assumptions
- The request is for a real project, not just an experiment.
- The team wants a practical plan that can be shared with others.

## Open Questions
- What is the exact deadline?
- Who is the target user or customer?
- What tools, data, or systems are already available?

## Task Breakdown
1. Clarify the full scope and success criteria.
2. Break the work into small deliverable steps.
3. Implement the first version.
4. Review the output and refine based on feedback.

## Risks
- Requirements may be incomplete or unclear.
- The timeline may be too short for the requested scope.
- Important details may be missing from th

In [4]:
from pathlib import Path
import json

BASE_DIR = Path("/kaggle/working/brief_to_plan_agent")
BASE_DIR.mkdir(parents=True, exist_ok=True)

SAMPLES_DIR = BASE_DIR / "samples"
SAMPLES_DIR.mkdir(exist_ok=True)

TEMPLATES_DIR = BASE_DIR / "templates"
TEMPLATES_DIR.mkdir(exist_ok=True)

sample_1 = """Launch a new customer onboarding flow for the website. Need it faster, less confusing, and ready by next month. Include sign-up, email verification, and basic analytics."""
sample_2 = """We need a better internal report for sales leads. Team wants something simple, reliable, and easy to share in weekly meetings."""

(SAMPLES_DIR / "sample_brief_1.txt").write_text(sample_1, encoding="utf-8")
(SAMPLES_DIR / "sample_brief_2.txt").write_text(sample_2, encoding="utf-8")

template = """# Execution Plan

## Summary

## Goal

## Key Facts

## Assumptions

## Open Questions

## Task Breakdown

## Risks

## Suggested Next Step
"""
(TEMPLATES_DIR / "plan_template.md").write_text(template, encoding="utf-8")

def load_sample_brief(name: str) -> str:
    path = SAMPLES_DIR / name
    return path.read_text(encoding="utf-8")

def load_plan_template() -> str:
    return (TEMPLATES_DIR / "plan_template.md").read_text(encoding="utf-8")

print("Created sample files:")
print(load_sample_brief("sample_brief_1.txt"))
print("\nPlan template:")
print(load_plan_template())

Created sample files:
Launch a new customer onboarding flow for the website. Need it faster, less confusing, and ready by next month. Include sign-up, email verification, and basic analytics.

Plan template:
# Execution Plan

## Summary

## Goal

## Key Facts

## Assumptions

## Open Questions

## Task Breakdown

## Risks

## Suggested Next Step



In [6]:
from pathlib import Path

SKILLS_BASE = BASE_DIR / "skills"
(SKILLS_BASE / "brief_analysis").mkdir(parents=True, exist_ok=True)
(SKILLS_BASE / "plan_generation").mkdir(parents=True, exist_ok=True)
(SKILLS_BASE / "review_check").mkdir(parents=True, exist_ok=True)

brief_analysis_skill = """---
name: brief-analysis
description: |
  Analyze messy project briefs, meeting notes, or feature requests.
  Use when the user wants a clean summary, key facts, assumptions, open questions, or a task breakdown.
  Do NOT use for code execution or deployment tasks.
---

# Brief Analysis Skill

## When to use
- The user gives an unclear project brief.
- The user needs the brief turned into structured facts.

## When not to use
- The user only wants code to run.
- The user already provided a complete spec.

## Workflow
1. Read the brief carefully.
2. Extract the goal, deadline, audience, and deliverables.
3. List assumptions and missing information.
4. Return structured notes for planning.
"""

plan_generation_skill = """---
name: plan-generation
description: |
  Generate a clear execution plan from extracted brief facts.
  Use when the agent needs a summary, task breakdown, risks, and next steps.
  Do NOT use for raw fact extraction.
---

# Plan Generation Skill

## When to use
- The brief has already been analyzed.
- The user wants a usable plan or project outline.

## When not to use
- The input has not yet been summarized.
- The task is only to identify facts.

## Workflow
1. Review extracted facts.
2. Convert them into a markdown execution plan.
3. Include tasks, risks, and next steps.
"""

review_check_skill = """---
name: review-check
description: |
  Review the generated execution plan for missing sections, clarity, and structural completeness.
  Use when the agent needs a quality check before showing the final plan.
  Do NOT use to generate the plan itself.
---

# Review Check Skill

## When to use
- The plan is already written.
- The agent needs a quality gate.

## When not to use
- The plan has not been generated yet.
- The task is only extraction or planning.

## Workflow
1. Check required sections.
2. Look for missing or weak parts.
3. Return review notes or confirm the plan is structurally sound.
"""

(SKILLS_BASE / "brief_analysis" / "SKILL.md").write_text(brief_analysis_skill, encoding="utf-8")
(SKILLS_BASE / "plan_generation" / "SKILL.md").write_text(plan_generation_skill, encoding="utf-8")
(SKILLS_BASE / "review_check" / "SKILL.md").write_text(review_check_skill, encoding="utf-8")

def load_skill(skill_name: str) -> str:
    path = SKILLS_BASE / skill_name / "SKILL.md"
    return path.read_text(encoding="utf-8")

print("Created skills:")
print(load_skill("brief_analysis")[:500])
print("\n---\n")
print(load_skill("plan_generation")[:500])
print("\n---\n")
print(load_skill("review_check")[:500])

Created skills:
---
name: brief-analysis
description: |
  Analyze messy project briefs, meeting notes, or feature requests.
  Use when the user wants a clean summary, key facts, assumptions, open questions, or a task breakdown.
  Do NOT use for code execution or deployment tasks.
---

# Brief Analysis Skill

## When to use
- The user gives an unclear project brief.
- The user needs the brief turned into structured facts.

## When not to use
- The user only wants code to run.
- The user already provided a comple

---

---
name: plan-generation
description: |
  Generate a clear execution plan from extracted brief facts.
  Use when the agent needs a summary, task breakdown, risks, and next steps.
  Do NOT use for raw fact extraction.
---

# Plan Generation Skill

## When to use
- The brief has already been analyzed.
- The user wants a usable plan or project outline.

## When not to use
- The input has not yet been summarized.
- The task is only to identify facts.

## Workflow
1. Review ex

In [7]:
def run_brief_to_plan_pipeline(brief: str) -> str:
    # Step 1: analyze the brief
    facts = extract_facts(brief)

    # Step 2: generate the plan from facts
    plan = plan_from_facts(facts)

    # Step 3: review the plan
    issues = review_plan(plan)

    # Step 4: attach review notes
    if issues:
        review_notes = "\n".join([f"- {issue}" for issue in issues])
    else:
        review_notes = "- No obvious structural issues found."

    final_output = plan + "\n## Final Review\n" + review_notes + "\n"
    return final_output


# Try it on your sample brief
print(run_brief_to_plan_pipeline(sample_brief))

# Execution Plan

## Summary
Launch a new customer onboarding flow for the website

## Goal
Improve the customer onboarding experience.

## Key Facts
- Keywords detected: customer onboarding, email workflow, analytics, sign-up flow
- Possible deadline: by next month
- Possible audience: customers
- Deliverables: basic analytics, email verification, sign-up flow

## Assumptions
- The request is for a real project, not just an experiment.
- The team wants a practical plan that can be shared with others.

## Open Questions
- What is the exact deadline?
- Who is the target user or customer?
- What tools, data, or systems are already available?

## Task Breakdown
1. Clarify the full scope and success criteria.
2. Break the work into small deliverable steps.
3. Implement the first version.
4. Review the output and refine based on feedback.

## Risks
- Requirements may be incomplete or unclear.
- The timeline may be too short for the requested scope.
- Important details may be missing from th

In [8]:
from IPython.display import Markdown, display
from pathlib import Path

OUTPUT_DIR = BASE_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

def demo_brief_to_plan(brief: str, output_name: str = "plan_output.md") -> str:
    final_plan = run_brief_to_plan_pipeline(brief)

    # Display nicely in notebook
    display(Markdown(final_plan))

    # Save to file for your repo / demo
    out_path = OUTPUT_DIR / output_name
    out_path.write_text(final_plan, encoding="utf-8")
    print(f"Saved to: {out_path}")

    return final_plan


# Try it with the sample brief
demo_brief_to_plan(sample_brief, "sample_plan_1.md")

# Execution Plan

## Summary
Launch a new customer onboarding flow for the website

## Goal
Improve the customer onboarding experience.

## Key Facts
- Keywords detected: customer onboarding, email workflow, analytics, sign-up flow
- Possible deadline: by next month
- Possible audience: customers
- Deliverables: basic analytics, email verification, sign-up flow

## Assumptions
- The request is for a real project, not just an experiment.
- The team wants a practical plan that can be shared with others.

## Open Questions
- What is the exact deadline?
- Who is the target user or customer?
- What tools, data, or systems are already available?

## Task Breakdown
1. Clarify the full scope and success criteria.
2. Break the work into small deliverable steps.
3. Implement the first version.
4. Review the output and refine based on feedback.

## Risks
- Requirements may be incomplete or unclear.
- The timeline may be too short for the requested scope.
- Important details may be missing from the brief.

## Suggested Next Step
Review the open questions with the user before starting implementation.

## Final Review
- No obvious structural issues found.


Saved to: /kaggle/working/brief_to_plan_agent/outputs/sample_plan_1.md


'# Execution Plan\n\n## Summary\nLaunch a new customer onboarding flow for the website\n\n## Goal\nImprove the customer onboarding experience.\n\n## Key Facts\n- Keywords detected: customer onboarding, email workflow, analytics, sign-up flow\n- Possible deadline: by next month\n- Possible audience: customers\n- Deliverables: basic analytics, email verification, sign-up flow\n\n## Assumptions\n- The request is for a real project, not just an experiment.\n- The team wants a practical plan that can be shared with others.\n\n## Open Questions\n- What is the exact deadline?\n- Who is the target user or customer?\n- What tools, data, or systems are already available?\n\n## Task Breakdown\n1. Clarify the full scope and success criteria.\n2. Break the work into small deliverable steps.\n3. Implement the first version.\n4. Review the output and refine based on feedback.\n\n## Risks\n- Requirements may be incomplete or unclear.\n- The timeline may be too short for the requested scope.\n- Importa

In [10]:
readme_text = """# Brief-to-Plan Agent

## What this project does
Brief-to-Plan Agent turns messy project briefs, meeting notes, or feature requests into a clean execution plan.

## Why this is useful
Teams often receive vague requests that are hard to turn into action. This agent helps by extracting the goal, identifying missing information, and producing a structured markdown plan with tasks, assumptions, risks, and next steps.

## How it works
1. The user pastes a brief.
2. The agent extracts key facts.
3. The agent generates a structured plan.
4. The agent reviews the result for completeness.
5. The final output is displayed and saved as markdown.

## Architecture
- Orchestrator: runs the full pipeline
- brief-analysis skill: extracts facts from the brief
- plan-generation skill: turns facts into a plan
- review-check skill: checks structure and completeness
- MCP-style tool layer: provides sample briefs and templates

## Course concepts used
- Agent / multi-agent style pipeline
- Agent skills
- Tool layer / MCP-style integration
- Security-minded human review
- Evaluation checks for output structure

## Files
- `specs/brief_to_plan_spec.md`
- `skills/brief_analysis/SKILL.md`
- `skills/plan_generation/SKILL.md`
- `skills/review_check/SKILL.md`
- `outputs/sample_plan_1.md`

## How to run
Run the notebook cells in order:
1. Spec and problem statement cells
2. Baseline plan generator
3. Review checker
4. Tool layer setup
5. Skill setup
6. End-to-end pipeline
7. Demo cell

## Evaluation
The project is checked by:
- verifying all required output sections exist
- testing multiple sample briefs
- checking that the plan stays structured and readable

## Limitations
This version uses simple logic for the first prototype. It can be improved later by adding stronger retrieval, richer agent reasoning, or a real UI.

## Next improvements
- add a richer front end
- add more sample briefs
- improve the review step
- connect the workflow to a stronger MCP server
"""

readme_path = BASE_DIR / "README.md"
readme_path.write_text(readme_text, encoding="utf-8")
print(f"README saved to: {readme_path}")
print(readme_text)

README saved to: /kaggle/working/brief_to_plan_agent/README.md
# Brief-to-Plan Agent

## What this project does
Brief-to-Plan Agent turns messy project briefs, meeting notes, or feature requests into a clean execution plan.

## Why this is useful
Teams often receive vague requests that are hard to turn into action. This agent helps by extracting the goal, identifying missing information, and producing a structured markdown plan with tasks, assumptions, risks, and next steps.

## How it works
1. The user pastes a brief.
2. The agent extracts key facts.
3. The agent generates a structured plan.
4. The agent reviews the result for completeness.
5. The final output is displayed and saved as markdown.

## Architecture
- Orchestrator: runs the full pipeline
- brief-analysis skill: extracts facts from the brief
- plan-generation skill: turns facts into a plan
- review-check skill: checks structure and completeness
- MCP-style tool layer: provides sample briefs and templates

## Course concept

In [11]:
test_briefs = {
    "onboarding": """
    Launch a new customer onboarding flow for the website. Need it faster, less confusing, and ready by next month. Include sign-up, email verification, and basic analytics.
    """,
    "sales_report": """
    We need a better internal report for sales leads. Team wants something simple, reliable, and easy to share in weekly meetings.
    """,
    "project_launch": """
    Plan a product launch for a new mobile app feature. We need marketing, support prep, release coordination, and a fallback plan.
    """,
    "vague_request": """
    Make this better.
    """,
    "support_workflow": """
    Improve our customer support workflow so the team can respond faster and reduce confusion. We need a clear process, templates, and escalation steps.
    """
}

required_sections = [
    "## Summary",
    "## Goal",
    "## Key Facts",
    "## Assumptions",
    "## Open Questions",
    "## Task Breakdown",
    "## Risks",
    "## Suggested Next Step",
    "## Final Review",
]

results = []

for name, brief in test_briefs.items():
    output = run_brief_to_plan_pipeline(brief)
    missing = [section for section in required_sections if section not in output]
    passed = len(missing) == 0

    results.append({
        "test_case": name,
        "passed": passed,
        "missing_sections": missing,
        "output_preview": output[:250]
    })

    print("=" * 80)
    print(f"TEST CASE: {name}")
    print("PASS" if passed else "FAIL")
    if missing:
        print("Missing sections:")
        for m in missing:
            print(f"- {m}")
    print("\nPreview:")
    print(output[:500])
    print()

# Save a simple evaluation report
report_lines = ["# Evaluation Report", ""]
for r in results:
    report_lines.append(f"## {r['test_case']}")
    report_lines.append(f"- Passed: {r['passed']}")
    if r["missing_sections"]:
        report_lines.append(f"- Missing: {', '.join(r['missing_sections'])}")
    else:
        report_lines.append("- Missing: None")
    report_lines.append("")

eval_path = OUTPUT_DIR / "evaluation_report.md"
eval_path.write_text("\n".join(report_lines), encoding="utf-8")

print("=" * 80)
print(f"Saved evaluation report to: {eval_path}")

TEST CASE: onboarding
PASS

Preview:
# Execution Plan

## Summary
Launch a new customer onboarding flow for the website

## Goal
Improve the customer onboarding experience.

## Key Facts
- Keywords detected: customer onboarding, email workflow, analytics, sign-up flow
- Possible deadline: by next month
- Possible audience: customers
- Deliverables: basic analytics, email verification, sign-up flow

## Assumptions
- The request is for a real project, not just an experiment.
- The team wants a practical plan that can be shared with o

TEST CASE: sales_report
PASS

Preview:
# Execution Plan

## Summary
We need a better internal report for sales leads

## Goal
Create a useful report or summary.

## Key Facts
- Keywords detected: report, sales workflow
- Possible deadline: Not stated
- Possible audience: internal team
- Deliverables: Not stated

## Assumptions
- The request is for a real project, not just an experiment.
- The team wants a practical plan that can be shared with others.

## O

# Project Summary

Brief-to-Plan Agent turns messy project briefs into clear execution plans.

## How it works
1. Paste a brief.
2. The agent extracts key facts.
3. The agent generates a markdown plan.
4. The agent checks the result for completeness.
5. The final output is displayed and saved.

## What this project demonstrates
- Agent-style pipeline
- Skills
- Tool layer
- Security-aware review
- Evaluation

# Architecture

The project uses a simple 3-step agent pipeline:

1. **Brief Analysis**  
   Extracts key facts from the input brief.

2. **Plan Generation**  
   Turns the extracted facts into a structured execution plan.

3. **Review Check**  
   Checks whether the final plan includes all required sections.

Supporting pieces:
- **Tool layer** for sample briefs and templates
- **Skills** for reusable task instructions
- **Evaluation** for testing multiple briefs
- **Security-aware review** so the agent stays structured and controlled

In [12]:
demo_brief = """
We need a better internal report for sales leads. Team wants something simple, reliable, and easy to share in weekly meetings.
"""

print(demo_brief_to_plan(demo_brief, "final_demo_plan.md"))

# Execution Plan

## Summary
We need a better internal report for sales leads

## Goal
Create a useful report or summary.

## Key Facts
- Keywords detected: report, sales workflow
- Possible deadline: Not stated
- Possible audience: internal team
- Deliverables: Not stated

## Assumptions
- The request is for a real project, not just an experiment.
- The team wants a practical plan that can be shared with others.

## Open Questions
- What is the exact deadline?
- Who is the target user or customer?
- What tools, data, or systems are already available?

## Task Breakdown
1. Clarify the full scope and success criteria.
2. Break the work into small deliverable steps.
3. Implement the first version.
4. Review the output and refine based on feedback.

## Risks
- Requirements may be incomplete or unclear.
- The timeline may be too short for the requested scope.
- Important details may be missing from the brief.

## Suggested Next Step
Review the open questions with the user before starting implementation.

## Final Review
- No obvious structural issues found.


Saved to: /kaggle/working/brief_to_plan_agent/outputs/final_demo_plan.md
# Execution Plan

## Summary
We need a better internal report for sales leads

## Goal
Create a useful report or summary.

## Key Facts
- Keywords detected: report, sales workflow
- Possible deadline: Not stated
- Possible audience: internal team
- Deliverables: Not stated

## Assumptions
- The request is for a real project, not just an experiment.
- The team wants a practical plan that can be shared with others.

## Open Questions
- What is the exact deadline?
- Who is the target user or customer?
- What tools, data, or systems are already available?

## Task Breakdown
1. Clarify the full scope and success criteria.
2. Break the work into small deliverable steps.
3. Implement the first version.
4. Review the output and refine based on feedback.

## Risks
- Requirements may be incomplete or unclear.
- The timeline may be too short for the requested scope.
- Important details may be missing from the brief.

## Sugg